# ACTIVA M1 — Steps 1–3: ActionFormer EPIC reference on Kaggle

**Before running:** Settings → Accelerator **GPU T4 x1** (or P100), Settings → **Internet ON**
(phone-verified Kaggle account needed for both).

What this notebook does:
1. Installs ActionFormer (pinned commit) and builds its NMS extension.
2. Gets the published EPIC-100 SlowFast features + pretrained verb/noun checkpoints
   (from an attached Kaggle dataset if present, otherwise downloads them from the README's Google Drive links).
3. **Step 2:** reproduces the published validation mAP (README: verb avg **23.51**, noun avg **21.88**).
4. **Step 3:** saves raw detections in seconds and prints one video's top detections next to its ground truth.

Outputs land in `/kaggle/working/`. These validation detections are for **checking code correctness only** —
do not tune thresholds on them (the 138 validation videos are Samsung evaluation data).

In [ ]:
!nvidia-smi
import sys, torch, numpy
print(sys.version); print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "numpy", numpy.__version__)

## 1. Install ActionFormer (pinned) and build NMS

In [ ]:
%cd /kaggle/working
!rm -rf actionformer_release
!git clone -q https://github.com/happyharrycn/actionformer_release.git
%cd /kaggle/working/actionformer_release
!git checkout -q 61ea7eb9308a568b0cf45e3804830836e30061de && git log --oneline -1
!pip install -q pyyaml pandas h5py joblib tensorboard gdown

In [ ]:
# ActionFormer targets numpy<=1.23; patch removed aliases (np.float/np.int/np.bool) so it runs on current numpy.
import re, pathlib
patched = []
for f in pathlib.Path("libs").rglob("*.py"):
    s = f.read_text()
    t = re.sub(r"\bnp\.(float|int|bool)\b(?![\d_])", r"\1", s)
    if t != s:
        f.write_text(t); patched.append(str(f))
print("patched:", patched)
# eval.py was written for torch 1.11: storage.cuda('cuda:0') fails on new torch, and torch>=2.6
# defaults to weights_only=True which rejects these checkpoints. Load them onto the device directly.
ev = pathlib.Path("eval.py"); s = ev.read_text()
old = "lambda storage, loc: storage.cuda(cfg['devices'][0])"
if old in s:
    ev.write_text(s.replace(old, "cfg['devices'][0], weights_only = False")); print("eval.py patched")

In [ ]:
%cd /kaggle/working/actionformer_release/libs/utils
!python setup.py install --user > /kaggle/working/nms_build.log 2>&1; tail -3 /kaggle/working/nms_build.log
%cd /kaggle/working/actionformer_release
!python -c "import torch, nms_1d_cpu; print('NMS extension OK')"

## 2. Get features + checkpoints
If you uploaded the three Google Drive files as a Kaggle dataset, attach it (Add Input) and they are found automatically.
Otherwise they are downloaded with `gdown` (the features archive is large; this can take a while).

In [ ]:
import glob, hashlib, os, shutil, subprocess, pathlib

DL = pathlib.Path("/tmp/downloads")  # keep big files out of the saved output
DL.mkdir(parents=True, exist_ok=True)
SOURCES = {  # from the ActionFormer README
    "features": ("1Z4U_dLuu6_cV5NBIrSzsSDOOj2Uar85X", "epic_kitchens"),
    "verb_ckpt": ("1Ta4ggKSj2YcszSrDbePlHe1ECF1CFKK4", "epic_slowfast_verb"),
    "noun_ckpt": ("1OTlxeiWj8JE9n1-LsRYogHmqgUdsE5PR", "epic_slowfast_noun"),
}
FEATURES_MD5 = "add9803756afd9a023bc9a9c547e0229"

def find_input(keyword):
    hits = [p for p in glob.glob("/kaggle/input/**/*", recursive=True) if keyword in os.path.basename(p)]
    return sorted(hits, key=len)[0] if hits else None

def get(name):
    file_id, keyword = SOURCES[name]
    hit = find_input(keyword)
    if hit:
        print(f"{name}: using attached input {hit}"); return pathlib.Path(hit)
    existing = sorted(p for p in DL.iterdir() if keyword in p.name)
    if existing:
        print(f"{name}: already downloaded {existing[0]}"); return existing[0]
    before = set(DL.iterdir())
    subprocess.run(["gdown", "--fuzzy", f"https://drive.google.com/file/d/{file_id}/view", "-O", str(DL) + "/"], check=True)
    new = sorted(set(DL.iterdir()) - before)
    if not new:
        raise RuntimeError(f"{name}: download produced no file (Drive quota? try uploading it as a Kaggle dataset)")
    print(f"{name}: downloaded {new[0]}"); return new[0]

def md5(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def unpack(path, dest):
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        return path
    shutil.unpack_archive(str(path), str(dest)); return dest

paths = {n: get(n) for n in SOURCES}
if paths["features"].is_file():
    got = md5(paths["features"]); print("features md5", got, "OK" if got == FEATURES_MD5 else "MISMATCH — re-download!")

In [ ]:
AF = pathlib.Path("/kaggle/working/actionformer_release")
EX = pathlib.Path("/tmp/extracted")  # /kaggle/working is capped at ~20 GB
feat_root = unpack(paths["features"], EX / "features")
verb_root = unpack(paths["verb_ckpt"], EX / "verb")
noun_root = unpack(paths["noun_ckpt"], EX / "noun")

def find_dir(root, must_contain):
    for d in [pathlib.Path(root), *pathlib.Path(root).rglob("*")]:
        if d.is_dir() and all((d / m).exists() for m in must_contain):
            return d
    raise FileNotFoundError(f"no dir under {root} containing {must_contain}")

def find_ckpt_dir(root):
    hits = sorted(pathlib.Path(root).rglob("*.pth.tar"))
    if not hits: raise FileNotFoundError(f"no .pth.tar under {root}")
    return hits[0].parent

epic_dir = find_dir(feat_root, ["annotations", "features"])
(AF / "data").mkdir(exist_ok=True)
link = AF / "data" / "epic_kitchens"
if link.exists() or link.is_symlink(): link.unlink()
link.symlink_to(epic_dir)

for kind, root in (("verb", verb_root), ("noun", noun_root)):
    dst = AF / "pretrained" / f"epic_slowfast_{kind}_reproduce"
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(find_ckpt_dir(root), dst)
    print(kind, "->", sorted(p.name for p in dst.iterdir()))
print("annotations:", sorted(p.name for p in (link / "annotations").iterdir()))
feats = sorted((link / "features").glob("*.npz")); print(len(feats), "feature files, e.g.", feats[0].name)

## 3. Step 2 — reproduce the published mAP
Expected (README, avg over tIoU 0.1–0.5): **verb 23.51**, **noun 21.88**. Small differences are fine; big ones mean wrong files/config.

In [ ]:
%cd /kaggle/working/actionformer_release
for kind in ("verb", "noun"):
    log = f"/kaggle/working/eval_{kind}.log"
    !python ./eval.py ./configs/epic_slowfast_{kind}.yaml ./pretrained/epic_slowfast_{kind}_reproduce/ > {log} 2>&1
    print(f"===== {kind} =====")
    !grep -iE "tiou|mAP|error|Traceback" {log} | tail -12

## 4. Step 3 — raw detections in seconds + one-video sanity check
`--saveonly` writes `eval_results.pkl` next to each checkpoint (ActionFormer already converts to seconds with
`(grid * 16 + 0.5 * 32) / 30`). We bundle both into `/kaggle/working/m1_raw_detections_val.pkl`.

In [ ]:
import pickle, json, numpy as np
raw = {}
for kind in ("verb", "noun"):
    !python ./eval.py ./configs/epic_slowfast_{kind}.yaml ./pretrained/epic_slowfast_{kind}_reproduce/ --saveonly > /kaggle/working/save_{kind}.log 2>&1
    with open(f"pretrained/epic_slowfast_{kind}_reproduce/eval_results.pkl", "rb") as f:
        r = pickle.load(f)
    vids = np.asarray(r["video-id"])
    raw[kind] = {v: {"segs": np.stack([r["t-start"][vids == v], r["t-end"][vids == v]], 1),
                     "labels": r["label"][vids == v], "scores": r["score"][vids == v]}
                 for v in np.unique(vids)}
    print(kind, len(raw[kind]), "videos,", len(vids), "detections")
with open("/kaggle/working/m1_raw_detections_val.pkl", "wb") as f:
    pickle.dump(raw, f)
print("saved /kaggle/working/m1_raw_detections_val.pkl")

In [ ]:
ann = {k: json.load(open(f"data/epic_kitchens/annotations/epic_kitchens_100_{k}.json"))["database"] for k in ("verb", "noun")}
vid = sorted(raw["verb"])[0]
meta = ann["verb"][vid]
print("video", vid, "duration", meta.get("duration"), "fps", meta.get("fps"))

f = np.load(f"data/epic_kitchens/features/{vid}.npz")
print("npz keys:", list(f.keys()), {k: f[k].shape for k in f.keys()})   # needed for Step 4 (T, dim)
T = f[list(f.keys())[0]].shape[0]
print("feature windows T =", T, "| expected ~", int((meta["duration"] * 30 - 32) // 16 + 1))

for kind in ("verb", "noun"):
    names = {a["label_id"]: a["label"] for v in ann[kind].values() for a in v.get("annotations", [])}
    d = raw[kind][vid]; top = np.argsort(-d["scores"])[:10]
    print(f"\n--- top-10 {kind} detections ---")
    for i in top:
        print(f"{d['segs'][i,0]:8.2f}-{d['segs'][i,1]:8.2f}s  {names.get(int(d['labels'][i]), d['labels'][i]):<14} {d['scores'][i]:.3f}")
    print(f"--- first 10 ground-truth {kind} segments ---")
    for a in sorted(ann[kind][vid]["annotations"], key=lambda a: a["segment"][0])[:10]:
        print(f"{a['segment'][0]:8.2f}-{a['segment'][1]:8.2f}s  {a['label']}")

## Done when
- Section 3 shows verb/noun average mAP close to 23.51 / 21.88.
- Section 4 detections line up with ground-truth times for the sample video.

Download `m1_raw_detections_val.pkl`, `eval_verb.log`, `eval_noun.log` from the Output panel and paste the
mAP lines + the Section 4 printout back to your assistant. Next: **Step 4**, the raw-MP4 feature gate.